## 04 - Logica Proposicional Conectivos e Permissivos
# Algoritmo de Intertravamento Preliminar da Linha de Produção

Implementação em Python dos blocos de permissivos (*Start Permissives*) e desarmes automáticos (*Trips*) para:
- **BC1**: Bomba Centrífuga de Alimentação
- **RC1**: Esteira Transportadora
- **VS2**: Válvula de Envase / Enchimento
- **VS3 / SL1**: Módulo de Inspeção de Nível
- **VS4 / AC1**: Atuador de Tampagem
- **VS5 / SFC1**: Módulo de Inspeção de Vedação

Avaliação lógica baseada no **vetor de estados global da planta**.

In [ ]:
from typing import Dict, Any
import pandas as pd

# Operadores Lógicos Proposicionais
def NOT(p: bool) -> bool:
    return not p

def AND(*args: bool) -> bool:
    return all(args)

def OR(*args: bool) -> bool:
    return any(args)

def XOR(p: bool, q: bool) -> bool:
    return p ^ q

print("Operadores lógicos carregados com sucesso.")

## 1. Blocos de Intertravamento e Permissivos Individuais

In [ ]:
def intertrava_BC1(sp_1_ok: bool, sq_1_ok: bool, ls_VS1_open: bool, 
                   p_AS1_high: bool, e_stop: bool, modo_valido: bool) -> Dict[str, bool]:
    trip = OR(NOT(sp_1_ok), NOT(sq_1_ok), NOT(ls_VS1_open), p_AS1_high, e_stop)
    permissivo = AND(sp_1_ok, sq_1_ok, ls_VS1_open, NOT(p_AS1_high), NOT(e_stop), modo_valido)
    return {"permissivo": permissivo, "trip": trip}

def intertrava_RC1(act_VS2: bool, ext_VS3: bool, ext_VS4: bool, 
                   ext_VS5: bool, e_stop: bool, modo_valido: bool) -> Dict[str, bool]:
    trip = OR(e_stop, act_VS2, ext_VS3, ext_VS4, ext_VS5)
    permissivo = AND(NOT(act_VS2), NOT(ext_VS3), NOT(ext_VS4), NOT(ext_VS5), NOT(e_stop), modo_valido)
    return {"permissivo": permissivo, "trip": trip}

def intertrava_VS2(s_pos_VS2: bool, cmd_RC1: bool, sp_2_ok: bool, 
                   e_stop: bool, modo_valido: bool) -> Dict[str, bool]:
    trip = OR(NOT(s_pos_VS2), cmd_RC1, NOT(sp_2_ok), e_stop)
    permissivo = AND(s_pos_VS2, NOT(cmd_RC1), sp_2_ok, NOT(e_stop), modo_valido)
    return {"permissivo": permissivo, "trip": trip}

def intertrava_VS3(s_pos_VS3: bool, cmd_RC1: bool, e_stop: bool, modo_valido: bool) -> Dict[str, bool]:
    trip = OR(NOT(s_pos_VS3), cmd_RC1, e_stop)
    permissivo = AND(s_pos_VS3, NOT(cmd_RC1), NOT(e_stop), modo_valido)
    return {"permissivo": permissivo, "trip": trip}

def intertrava_VS4(s_pos_VS4: bool, aprov_nivel: bool, cmd_RC1: bool, 
                   e_stop: bool, modo_valido: bool) -> Dict[str, bool]:
    trip = OR(NOT(s_pos_VS4), NOT(aprov_nivel), cmd_RC1, e_stop)
    permissivo = AND(s_pos_VS4, aprov_nivel, NOT(cmd_RC1), NOT(e_stop), modo_valido)
    return {"permissivo": permissivo, "trip": trip}

def intertrava_VS5(s_pos_VS5: bool, cmd_RC1: bool, e_stop: bool, modo_valido: bool) -> Dict[str, bool]:
    trip = OR(NOT(s_pos_VS5), cmd_RC1, e_stop)
    permissivo = AND(s_pos_VS5, NOT(cmd_RC1), NOT(e_stop), modo_valido)
    return {"permissivo": permissivo, "trip": trip}

print("Blocos individuais configurados com sucesso.")

## 2. Avaliador Global do Vetor de Estados da Planta

In [ ]:
def avaliar_vetor_estados(planta: Dict[str, bool]) -> Dict[str, Any]:
    modo_valido = XOR(planta.get("auto_mode", False), planta.get("manual_mode", False))
    e_stop = planta.get("e_stop", False)
    cmd_RC1 = planta.get("cmd_RC1", False)
    
    bc1 = intertrava_BC1(planta.get("sp_1_ok", False), planta.get("sq_1_ok", False), planta.get("ls_VS1_open", False), planta.get("p_AS1_high", False), e_stop, modo_valido)
    rc1 = intertrava_RC1(planta.get("act_VS2", False), planta.get("ext_VS3", False), planta.get("ext_VS4", False), planta.get("ext_VS5", False), e_stop, modo_valido)
    vs2 = intertrava_VS2(planta.get("s_pos_VS2", False), cmd_RC1, planta.get("sp_2_ok", False), e_stop, modo_valido)
    vs3 = intertrava_VS3(planta.get("s_pos_VS3", False), cmd_RC1, e_stop, modo_valido)
    
    aprov_nivel = AND(planta.get("ext_VS3", False), planta.get("sl1_ok", False))
    vs4 = intertrava_VS4(planta.get("s_pos_VS4", False), planta.get("aprov_nivel", aprov_nivel), cmd_RC1, e_stop, modo_valido)
    
    vs5 = intertrava_VS5(planta.get("s_pos_VS5", False), cmd_RC1, e_stop, modo_valido)
    aprov_vedacao = AND(planta.get("ext_VS5", False), planta.get("sfc_1", False))

    return {
        "Modo_Valido": modo_valido,
        "Perm_BC1": bc1["permissivo"], "Trip_BC1": bc1["trip"],
        "Perm_RC1": rc1["permissivo"], "Trip_RC1": rc1["trip"],
        "Perm_VS2": vs2["permissivo"], "Trip_VS2": vs2["trip"],
        "Perm_VS3": vs3["permissivo"], "Trip_VS3": vs3["trip"],
        "Aprov_Nivel": aprov_nivel,
        "Perm_VS4": vs4["permissivo"], "Trip_VS4": vs4["trip"],
        "Perm_VS5": vs5["permissivo"], "Trip_VS5": vs5["trip"],
        "Aprov_Vedacao": aprov_vedacao
    }

## 3. Testes e Simulação dos Vetores de Estado

In [ ]:
cenarios = [
    {
        "Cenario": "Operacao Normal - Bomba BC1 Pronta",
        "vetor": {"auto_mode": True, "manual_mode": False, "e_stop": False, "sp_1_ok": True, "sq_1_ok": True, "ls_VS1_open": True, "p_AS1_high": False, "cmd_RC1": False, "act_VS2": False, "ext_VS3": False, "ext_VS4": False, "ext_VS5": False}
    },
    {
        "Cenario": "Emergencia Geral (E-STOP Ativo)",
        "vetor": {"auto_mode": True, "manual_mode": False, "e_stop": True, "sp_1_ok": True, "sq_1_ok": True, "ls_VS1_open": True, "p_AS1_high": False, "cmd_RC1": False, "act_VS2": False, "ext_VS3": False, "ext_VS4": False, "ext_VS5": False}
    },
    {
        "Cenario": "Esteira em Movimento (Bloqueia Estacoes)",
        "vetor": {"auto_mode": True, "manual_mode": False, "e_stop": False, "cmd_RC1": True, "s_pos_VS2": True, "sp_2_ok": True, "s_pos_VS3": True, "s_pos_VS4": True, "aprov_nivel": True, "s_pos_VS5": True}
    },
    {
        "Cenario": "Atuador VS3 Estendido (Bloqueia Esteira)",
        "vetor": {"auto_mode": True, "manual_mode": False, "e_stop": False, "cmd_RC1": False, "act_VS2": False, "ext_VS3": True, "ext_VS4": False, "ext_VS5": False}
    }
]

resultados = []
for c in cenarios:
    res = avaliar_vetor_estados(c["vetor"])
    resultados.append({
        "Cenario": c["Cenario"],
        "Perm BC1": res["Perm_BC1"], "Trip BC1": res["Trip_BC1"],
        "Perm RC1": res["Perm_RC1"], "Trip RC1": res["Trip_RC1"],
        "Perm VS2": res["Perm_VS2"], "Trip VS2": res["Trip_VS2"],
        "Perm VS4": res["Perm_VS4"], "Trip VS4": res["Trip_VS4"]
    })

pd.DataFrame(resultados)